# 00 · Data Loading & Verification
Load `data_pipeline` processed files (2015-2025, 6204 rows) and create `ml_base` for ML.


In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../../data_pipeline/data/processed")

# Expected counties and files
COUNTIES = ["Baringo","Bomet","Bungoma","Busia","Elgeyo Marakwet","Embu","Garissa","Homa Bay","Isiolo","Kajiado","Kakamega","Kericho","Kiambu","Kilifi","Kirinyaga","Kisii","Kisumu","Kitui","Kwale","Laikipia","Lamu","Machakos","Makueni","Mandera","Marsabit","Meru","Migori","Mombasa","Murang'a","Nairobi","Nakuru","Nandi","Narok","Nyamira","Nyandarua","Nyeri","Samburu","Siaya","Taita Taveta","Tana River","Tharaka Nithi","Trans Nzoia","Turkana","Uasin Gishu","Vihiga","Wajir","West Pokot"]

for f in ["NDVI_FINAL.csv","Weather_FINAL.csv","Demographic_Crop_Activity_FINAL.csv","master_cleaned_2015_2025.csv"]:
    p = DATA_DIR / f
    print(f"{'✓' if p.exists() else '✗'} {f} ({p.stat().st_size/1024:.0f} KB)" if p.exists() else f"✗ {f} missing")
print(f"Counties: {len(COUNTIES)}")


### Load and build master


In [ ]:
# Load sources - master_cleaned is source of truth, otherwise join NDVI+Weather+Demo
master_path = DATA_DIR / "master_cleaned_2015_2025.csv"
if master_path.exists():
    master = pd.read_csv(master_path)
    # Normalize names for ML (temp_mean_c etc)
    master = master.rename(columns={"temperature_mean":"temp_mean_c","rainfall_sum":"rainfall_mm","soil_moisture_mean":"soil_moisture"})
else:
    ndvi = pd.read_csv(DATA_DIR / "NDVI_FINAL.csv")
    weather = pd.read_csv(DATA_DIR / "Weather_FINAL.csv")
    demo = pd.read_csv(DATA_DIR / "Demographic_Crop_Activity_FINAL.csv")
    master = ndvi.merge(weather, on=["county","year","month"], how="inner")
    master = master.merge(demo, on="county", how="left")
    master = master.rename(columns={"temperature_mean":"temp_mean_c","rainfall_sum":"rainfall_mm","soil_moisture_mean":"soil_moisture"})

master["year_month"] = master["year"].astype(str) + "-" + master["month"].astype(str).str.zfill(2)
print(f"Master: {master.shape} | {master['county'].nunique()} counties | {master['year'].min()}-{master['year'].max()}")
display(master.head())


### Verify and save ml_base


In [ ]:
# Verify counties
missing = set(COUNTIES) - set(master["county"].unique())
assert not missing, f"Missing: {missing}"
extra = set(master["county"].unique()) - set(COUNTIES)
assert not extra, f"Extra: {extra}"

# Balanced panel check (no hardcode 132, just balanced)
cnt = master.groupby("county").size()
assert cnt.nunique() == 1, f"Unbalanced: {cnt.unique()}"
print(f"Balanced: {cnt.iloc[0]} months per county")

# Fill missing climate values (if any) with monthly national mean
for col in ["temp_mean_c","rainfall_mm","soil_moisture","ndvi_mean"]:
    if col in master.columns and master[col].isna().any():
        master[col] = master.groupby(["year","month"])[col].transform(lambda s: s.fillna(s.mean()))
        master[col] = master[col].fillna(master[col].mean())

assert not master.isna().any().any(), "Still has NA"
print(f"Missing after fill: {master.isna().sum().sum()}")

# Save ml_base for next notebooks
out = DATA_DIR / "ml_base_2015_2025.csv"
master.to_csv(out, index=False)
print(f"Saved {out} ({len(master)} rows)")
